In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import torch 
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

import rasterio

In [18]:
#set paths 
base_dir = "/Users/sofiasuhinin/Desktop/Subglacial_Lakes_Data"
lake_location_path = os.path.join(base_dir, "lake_locations.csv")
lake_and_null_path = os.path.join(base_dir, "lake_and_null_loc.csv")
dataset_split_path = os.path.join(base_dir, "dataset_split.csv")
training_set = os.path.join(base_dir, "lakeTS")

lake_location = pd.read_csv(lake_location_path)
lake_and_null = pd.read_csv(lake_and_null_path)
dataset_split = pd.read_csv(dataset_split_path)

In [ ]:
# list all netcdf files
nc_files = sorted([f for f in os.listdir(training_set) if f.endswith(".nc")])

file_df = pd.DataFrame({"file_name": nc_files})
file_df["name"] = file_df["file_name"].str.replace(".nc", "", regex=False)
file_df["file_path"] = file_df["file_name"].apply(lambda x: os.path.join(training_set, x))

print(file_df.head())
print("Total .nc files:", len(file_df))

      file_name       name                                          file_path
0    ANZAC19.nc    ANZAC19  /Users/sofiasuhinin/Desktop/Subglacial_Lakes_D...
1  Academy45.nc  Academy45  /Users/sofiasuhinin/Desktop/Subglacial_Lakes_D...
2  Academy91.nc  Academy91  /Users/sofiasuhinin/Desktop/Subglacial_Lakes_D...
3    Adams17.nc    Adams17  /Users/sofiasuhinin/Desktop/Subglacial_Lakes_D...
4    Adams19.nc    Adams19  /Users/sofiasuhinin/Desktop/Subglacial_Lakes_D...
Total .nc files: 105


In [33]:
# get set of known lake names
lake_names = set(lake_location["name"].astype(str))

# label files
file_df["label"] = file_df["name"].isin(lake_names).astype(int)

In [34]:
dataset_split["file_name"] = (
    dataset_split["file"]
    .astype(str)
    .str.strip()
    .str.replace("\\", "/", regex=False)
    .str.split("/")
    .str[-1]
)

print(dataset_split[["file","file_name","dataset"]].head())

                                                file       file_name dataset
0  /Users/sofiasuhinin/Desktop/Subglacial_Lakes_D...       null_8.nc   train
1  /Users/sofiasuhinin/Desktop/Subglacial_Lakes_D...      Totten7.nc   train
2  /Users/sofiasuhinin/Desktop/Subglacial_Lakes_D...  Institute14.nc   train
3  /Users/sofiasuhinin/Desktop/Subglacial_Lakes_D...     Totten82.nc   train
4  /Users/sofiasuhinin/Desktop/Subglacial_Lakes_D...      Amery22.nc   train


In [41]:
df = file_df.merge(
    dataset_split[["file_name","dataset"]],
    on="file_name",
    how="left"
)

print(df["dataset"].value_counts())

train_df = df[df["dataset"] == "train"].reset_index(drop=True)
val_df   = df[df["dataset"] == "val"].reset_index(drop=True)
test_df  = df[df["dataset"] == "test"].reset_index(drop=True)

dataset
train    73
test     16
val      16
Name: count, dtype: int64


In [52]:
class LakeDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["file_path"]
        label = np.float32(row["label"])

        ds = xr.open_dataset(path)
        arr = ds["anomaly"].values  # shape: (time, y, x)
        ds.close()

        arr = np.asarray(arr, dtype=np.float32)

        if np.all(np.isnan(arr)):
            mean_img = np.zeros((arr.shape[1], arr.shape[2]), dtype=np.float32)
            std_img = np.zeros((arr.shape[1], arr.shape[2]), dtype=np.float32)
        else:
            mean_img = np.nanmean(arr, axis=0)
            std_img = np.nanstd(arr, axis=0)

            # Replace remaining NaNs/Infs
            mean_img = np.nan_to_num(mean_img, nan=0.0, posinf=0.0, neginf=0.0)
            std_img = np.nan_to_num(std_img, nan=0.0, posinf=0.0, neginf=0.0)

        x = np.stack([mean_img, std_img], axis=0)

        # Normalize each channel safely
        for c in range(x.shape[0]):
            mu = np.mean(x[c])
            sigma = np.std(x[c])

            if sigma < 1e-6:
                x[c] = x[c] - mu
            else:
                x[c] = (x[c] - mu) / sigma

        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        return torch.tensor(x, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

In [53]:

train_dataset = LakeDataset(train_df)
val_dataset   = LakeDataset(val_df)
test_dataset  = LakeDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=2)
test_loader  = DataLoader(test_dataset, batch_size=2)

In [ ]:
X, y = train_dataset[0]

print("Input shape:", X.shape)
print("Label:", y)

/var/folders/3r/pr708byx3wq5670k7_scfc240000gn/T/ipykernel_87570/1523333800.py:23: RuntimeWarning: Mean of empty slice
  mean_img = np.nanmean(arr, axis=0)


Input shape: torch.Size([2, 501, 501])
Label: tensor(1.)
torch.Size([2, 2, 501, 501])
torch.Size([2])


In [58]:
X_batch, y_batch = next(iter(train_loader))
print(X_batch.shape)
print(y_batch.shape)

/var/folders/3r/pr708byx3wq5670k7_scfc240000gn/T/ipykernel_87570/1523333800.py:23: RuntimeWarning: Mean of empty slice
  mean_img = np.nanmean(arr, axis=0)


torch.Size([2, 2, 501, 501])
torch.Size([2])


In [59]:

class LakeCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 501 -> 250

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 250 -> 125

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 125 -> 62

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 62 -> 31

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [60]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LakeCNN().to(device)
print(device)
print(model)

cpu
LakeCNN(
  (features): Sequential(
    (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_featu

In [61]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LakeCNN().to(device)
print(device)
print(model)

cpu
LakeCNN(
  (features): Sequential(
    (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_featu

In [67]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LakeCNN().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Device:", device)

Device: cpu


In [68]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for X, y in loader:
        X = X.to(device)
        y = y.to(device).unsqueeze(1)  # shape: [batch, 1]

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

    return total_loss / len(loader.dataset)

In [69]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X = X.to(device)
        y = y.to(device).unsqueeze(1)

        logits = model(X)
        loss = criterion(logits, y)

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        total_loss += loss.item() * X.size(0)
        correct += (preds == y).sum().item()
        total += y.numel()

    avg_loss = total_loss / len(loader.dataset)
    acc = correct / total
    return avg_loss, acc

In [70]:
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

/var/folders/3r/pr708byx3wq5670k7_scfc240000gn/T/ipykernel_87570/1523333800.py:23: RuntimeWarning: Mean of empty slice
  mean_img = np.nanmean(arr, axis=0)
/opt/anaconda3/envs/greenlandMapping/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


RuntimeError: stack expects each tensor to be equal size, but got [2, 500, 500] at entry 0 and [2, 501, 501] at entry 1

In [71]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

/var/folders/3r/pr708byx3wq5670k7_scfc240000gn/T/ipykernel_87570/1523333800.py:23: RuntimeWarning: Mean of empty slice
  mean_img = np.nanmean(arr, axis=0)


RuntimeError: stack expects each tensor to be equal size, but got [2, 501, 501] at entry 0 and [2, 501, 500] at entry 1